<div dir="rtl">
  <p style="color: #80f9fa; text-align: right;">
    <b>بلوک inception</b>
  </p>
</div>

---

In [1]:
import torch
import torch.nn as nn

class InceptionBlock(nn.Module):
    def __init__(self, in_channels, ch1x1, ch3x3red, ch3x3, ch5x5red, ch5x5, pool_proj):
        super(InceptionBlock, self).__init__()
        
        # 1x1 convolution
        self.branch1 = nn.Conv2d(in_channels, ch1x1, kernel_size=1)
        
        # 1x1 followed by 3x3 convolution
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, ch3x3red, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(ch3x3red, ch3x3, kernel_size=3, padding=1),
        )

        # 1x1 followed by 5x5 convolution
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, ch5x5red, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(ch5x5red, ch5x5, kernel_size=5, padding=2),
        )

        # 3x3 max pooling followed by 1x1 conv
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels, pool_proj, kernel_size=1),
        )

    def forward(self, x):
        out1 = self.branch1(x)
        out2 = self.branch2(x)
        out3 = self.branch3(x)
        out4 = self.branch4(x)
        return torch.cat([out1, out2, out3, out4], dim=1)  # Concatenate along channels

In [2]:
import torch

x = torch.randn(1, 3, 32, 32)

model = InceptionBlock(
    in_channels=3,
    ch1x1=16, 
    ch3x3red=8, ch3x3=16, 
    ch5x5red=4, ch5x5=8, 
    pool_proj=8
)

y = model(x)
print(y.shape)

torch.Size([1, 48, 32, 32])


<div dir="rtl" style="text-align: right;">
تمرین:
یک شبکه عصبی کانولوشنی برای داده‌ی
mnist
براساس معماری زیر و مطابق با قالب پیوست، آموزش بدهید.
لازم به ذکر است، تنها نیاز به جایگذاری محتوای بلوک
residual
با یک بلوک
inception
است.

معماری پیشنهادی

- بلوک Sequential:
    - Conv2D(32, 3×3, padding=1) → ReLU → MaxPool(2×2)

- (Residual Block):
    - Conv2D(32, 3×3, padding=1) → BatchNorm → ReLU
    - Conv2D(64, 3×3, padding=1) → BatchNorm → ReLU
    - Conv2D(32, 3×3, padding=1) → BatchNorm → ReLU

- (Inception Block):
    - InceptionBlock(
                in_channels=32,
                ch1x1=16, 
                ch3x3red=8, ch3x3=16, 
                ch5x5red=4, ch5x5=8, 
                pool_proj=8
        )

- لایه‌های Fully Connected (Dense) در مدل Sequential:
    - Flatten()
    - Dense(512) → ReLU → Dropout(0.4)
    - Dense(256) → ReLU → Dropout(0.4)
    - Dense(128) → ReLU
    - Dense(10)
</div>

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class InceptionBlock(nn.Module):
    def __init__(self, in_channels, ch1x1, ch3x3red, ch3x3, ch5x5red, ch5x5, pool_proj):
        super(InceptionBlock, self).__init__()
        
        # 1x1 convolution
        self.branch1 = nn.Conv2d(in_channels, ch1x1, kernel_size=1)
        
        # 1x1 followed by 3x3 convolution
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, ch3x3red, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(ch3x3red, ch3x3, kernel_size=3, padding=1),
        )

        # 1x1 followed by 5x5 convolution
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, ch5x5red, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(ch5x5red, ch5x5, kernel_size=5, padding=2),
        )

        # 3x3 max pooling followed by 1x1 conv
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels, pool_proj, kernel_size=1),
        )

    def forward(self, x):
        out1 = self.branch1(x)
        out2 = self.branch2(x)
        out3 = self.branch3(x)
        out4 = self.branch4(x)
        return torch.cat([out1, out2, out3, out4], dim=1)  # Concatenate along channels


class Model(nn.Module):
    def __init__(self, num_classes=10):
        super(Model, self).__init__()
        

        self.conv_block1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding="same"),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        self.conv_block2 = nn.Sequential(
                    nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding="same"),
                    nn.BatchNorm2d(32),
                    nn.ReLU(),
                    nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding="same"),
                    nn.BatchNorm2d(64),
                    nn.ReLU(),
                    nn.Conv2d(in_channels=64, out_channels=32, kernel_size=3, padding="same"),
                    nn.BatchNorm2d(32),
                    nn.ReLU()
                )

        self.conv_block3 = InceptionBlock(
                in_channels=32,
                ch1x1=16, 
                ch3x3red=8, ch3x3=16, 
                ch5x5red=4, ch5x5=8, 
                pool_proj=8
        )


        self.fc_block = nn.Sequential(
            nn.Flatten(),
            nn.Linear(48 * 14 * 14, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = x + self.conv_block2(x)
        print(x.shape)
        x = self.conv_block3(x)
        print(x.shape)
        x = self.fc_block(x)
        return x

x = torch.randn(1, 1, 28, 28)
model = Model()
y = model(x)
print(y.shape)

torch.Size([1, 32, 14, 14])
torch.Size([1, 48, 14, 14])
torch.Size([1, 10])
